# ChatGPT Archive Compiler — budgeted semantic atlas

This is the cost-controlled revision of the third notebook. It transforms the validated Archive IR into a **semantic atlas** of subjects, projects, purposes, recurring themes, relationships, and long-running intellectual threads, then renders a thematic book.

The complete archive is still represented, embedded, graphed, categorized, and listed. The crucial change is that external structured analysis is **selective and hierarchical**: transparent local code creates a compact baseline profile for every conversation, while only a bounded, graph-selected set of central and time-spanning representatives receives model-written refinement. Category naming and archive synthesis operate on those compressed profiles rather than thousands of full conversations.

Every API request must reserve its configured worst-case input/output cost in a persistent ledger **before transmission**. The default cumulative ceiling is **$5.00**. Interrupted or unobservable requests retain their conservative reservation, so restarting Colab cannot reset the authorization. The notebook remains disabled until local preflight proves that the scheduled plan fits the ceiling and you enter separate privacy and spending acknowledgments.

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

## Configuration

The defaults target a low-single-digit first-pass cost without reducing the archive to a purely chronological index. `text-embedding-3-small` supplies semantic geometry for every bounded conversation. `gpt-5.6-luna` refines at most three representatives per discovered category and interprets the taxonomy. One `gpt-5.6-terra` request writes the final category/project synthesis from compressed evidence.

The expected cost and the more conservative scheduled reservation are different. Expected cost uses realistic output assumptions; scheduled reservation assumes every request consumes its full configured output ceiling. The run is refused unless that conservative plan fits `HARD_API_BUDGET_USD`, and the persistent request ledger enforces the same ceiling during execution.

The book retains the navy-and-copper editorial system, book typography, trade/A4 geometry, running furniture, cross-references, project timelines, and a complete category directory. Expanded prose remains selective so a large archive produces a useful book rather than an unbounded typesetting job.

In [ ]:
from pathlib import Path

REPO_BRANCH = "agent/rebuild-colab-workflow"  # @param {type:"string"}
REPO_FULL_NAME = "jcollins-bioinfo/chatgpt-archive-compiler"
PUBLIC_REPO_URL = f"https://github.com/{REPO_FULL_NAME}.git"
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/ChatGPT Data Export")
OUTPUT_ROOT = DRIVE_PROJECT_DIR / "outputs" / "semantic"

ANALYSIS_MODE = "budgeted"  # @param ["budgeted", "local"]
EMBEDDING_MODEL = "text-embedding-3-small"  # @param {type:"string"}
EMBEDDING_DIMENSIONS = 512  # @param {type:"integer"}
EMBEDDING_MAX_TOKENS_PER_ITEM = 4000  # @param {type:"integer"}
PROFILE_MODEL = "gpt-5.6-luna"  # @param {type:"string"}
TAXONOMY_MODEL = "gpt-5.6-luna"  # @param {type:"string"}
SYNTHESIS_MODEL = "gpt-5.6-terra"  # @param {type:"string"}
PROFILE_REASONING_EFFORT = "low"  # @param ["none", "low", "medium"]
TAXONOMY_REASONING_EFFORT = "medium"  # @param ["low", "medium", "high"]
SYNTHESIS_REASONING_EFFORT = "high"  # @param ["medium", "high", "xhigh"]
PROFILE_MAX_OUTPUT_TOKENS = 6000  # @param {type:"integer"}
TAXONOMY_MAX_OUTPUT_TOKENS = 16000  # @param {type:"integer"}
SYNTHESIS_MAX_OUTPUT_TOKENS = 24000  # @param {type:"integer"}

MAX_CHARACTERS_PER_CONVERSATION = 16000  # @param {type:"integer"}
EMBEDDING_BATCH_SIZE = 96  # @param {type:"integer"}
EMBEDDING_TOKENS_PER_MINUTE = 800000  # @param {type:"integer"}
ANALYSIS_BATCH_SIZE = 8  # @param {type:"integer"}
NEAREST_NEIGHBORS = 12  # @param {type:"integer"}
MIN_SIMILARITY = 0.32  # @param {type:"number"}
MAX_LEAF_CATEGORIES = 48  # @param {type:"integer"}
REFINED_CONVERSATIONS_PER_CATEGORY = 3  # @param {type:"integer"}
MAX_REFINED_CONVERSATIONS = 144  # @param {type:"integer"}
REVIEW_CONFIDENCE_THRESHOLD = 0.65  # @param {type:"number"}
MAX_CONVERSATIONS = 0  # @param {type:"integer"}

# Official OpenAI prices observed on this date; re-verify if running materially later.
PRICE_SNAPSHOT_DATE = "2026-07-19"
EMBEDDING_INPUT_USD_PER_MILLION = 0.02  # @param {type:"number"}
PROFILE_INPUT_USD_PER_MILLION = 1.00  # @param {type:"number"}
PROFILE_OUTPUT_USD_PER_MILLION = 6.00  # @param {type:"number"}
TAXONOMY_INPUT_USD_PER_MILLION = 1.00  # @param {type:"number"}
TAXONOMY_OUTPUT_USD_PER_MILLION = 6.00  # @param {type:"number"}
SYNTHESIS_INPUT_USD_PER_MILLION = 2.50  # @param {type:"number"}
SYNTHESIS_OUTPUT_USD_PER_MILLION = 15.00  # @param {type:"number"}
HARD_API_BUDGET_USD = 5.00  # @param {type:"number"}
SAFE_RESUME_SOURCE_COMMITS = {
    "0ee3b30481ba0c7939665025bb2d6b0ed30e9cbd",
    "6d8a85f0f8c59822aeb1a5e781388816b34b44f9",
}

BOOK_TITLE = "A Semantic Atlas of My ChatGPT Archive"  # @param {type:"string"}
BOOK_SUBTITLE = (
    "Subjects, projects, connections, and intellectual trajectories"  # @param {type:"string"}
)
BOOK_AUTHOR = ""  # @param {type:"string"}
PAPER_SIZE = "trade"  # @param ["trade", "a4"]
MAX_EXPANDED_BOOK_PROFILES = 240  # @param {type:"integer"}
MAX_PROJECT_TIMELINES = 24  # @param {type:"integer"}
MAX_TIMELINE_EVENTS_PER_PROJECT = 12  # @param {type:"integer"}
RENDER_PDF = True  # @param {type:"boolean"}

if not DRIVE_PROJECT_DIR.is_dir():
    raise RuntimeError("Expected Drive folder is missing: MyDrive/ChatGPT Data Export")
if not REPO_BRANCH.strip():
    raise ValueError("REPO_BRANCH must not be empty.")
if ANALYSIS_MODE not in {"budgeted", "local"}:
    raise ValueError("ANALYSIS_MODE must be 'budgeted' or 'local'.")
if not all(
    model.strip() for model in (EMBEDDING_MODEL, PROFILE_MODEL, TAXONOMY_MODEL, SYNTHESIS_MODEL)
):
    raise ValueError("Model identifiers must not be empty.")
if PROFILE_REASONING_EFFORT not in {"none", "low", "medium"}:
    raise ValueError("Unsupported PROFILE_REASONING_EFFORT.")
if TAXONOMY_REASONING_EFFORT not in {"low", "medium", "high"}:
    raise ValueError("Unsupported TAXONOMY_REASONING_EFFORT.")
if SYNTHESIS_REASONING_EFFORT not in {"medium", "high", "xhigh"}:
    raise ValueError("Unsupported SYNTHESIS_REASONING_EFFORT.")
if (
    min(
        EMBEDDING_DIMENSIONS,
        EMBEDDING_MAX_TOKENS_PER_ITEM,
        MAX_CHARACTERS_PER_CONVERSATION,
        EMBEDDING_BATCH_SIZE,
        EMBEDDING_TOKENS_PER_MINUTE,
        ANALYSIS_BATCH_SIZE,
        NEAREST_NEIGHBORS,
        MAX_LEAF_CATEGORIES,
        REFINED_CONVERSATIONS_PER_CATEGORY,
        MAX_REFINED_CONVERSATIONS,
        MAX_EXPANDED_BOOK_PROFILES,
        MAX_PROJECT_TIMELINES,
        MAX_TIMELINE_EVENTS_PER_PROJECT,
        PROFILE_MAX_OUTPUT_TOKENS,
        TAXONOMY_MAX_OUTPUT_TOKENS,
        SYNTHESIS_MAX_OUTPUT_TOKENS,
    )
    <= 0
):
    raise ValueError("Dimensions, limits, and batch sizes must be positive.")
if MAX_LEAF_CATEGORIES > 512:
    raise ValueError("MAX_LEAF_CATEGORIES must not exceed 512.")
if MAX_REFINED_CONVERSATIONS > 4096:
    raise ValueError("MAX_REFINED_CONVERSATIONS must not exceed 4096.")
if not -1.0 <= MIN_SIMILARITY <= 1.0:
    raise ValueError("MIN_SIMILARITY must be between -1 and 1.")
if not 0.0 <= REVIEW_CONFIDENCE_THRESHOLD <= 1.0:
    raise ValueError("REVIEW_CONFIDENCE_THRESHOLD must be between zero and one.")
if MAX_CONVERSATIONS < 0:
    raise ValueError("MAX_CONVERSATIONS must be zero or positive.")
if (
    min(
        EMBEDDING_INPUT_USD_PER_MILLION,
        PROFILE_INPUT_USD_PER_MILLION,
        PROFILE_OUTPUT_USD_PER_MILLION,
        TAXONOMY_INPUT_USD_PER_MILLION,
        TAXONOMY_OUTPUT_USD_PER_MILLION,
        SYNTHESIS_INPUT_USD_PER_MILLION,
        SYNTHESIS_OUTPUT_USD_PER_MILLION,
        HARD_API_BUDGET_USD,
    )
    <= 0
):
    raise ValueError("Prices and HARD_API_BUDGET_USD must be positive.")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Repository branch: {REPO_BRANCH}")
print(f"Analysis mode: {ANALYSIS_MODE}")
print(f"Embedding model: {EMBEDDING_MODEL} ({EMBEDDING_DIMENSIONS} dimensions)")
print(f"Embedding pace: {EMBEDDING_TOKENS_PER_MINUTE:,} tokens/minute")
print(f"Profile / taxonomy / synthesis: {PROFILE_MODEL} / {TAXONOMY_MODEL} / {SYNTHESIS_MODEL}")
print(f"Maximum external profiles: {MAX_REFINED_CONVERSATIONS}")
print(f"Hard configured API budget: ${HARD_API_BUDGET_USD:.2f}")

## Reproducible ephemeral checkout

Add two private Colab secrets and enable **Notebook access** for each:

- `GITHUB_TOKEN`: fine-grained token with read-only **Contents** access to the private repository.
- `OPENAI_API_KEY`: read only after the local cost plan fits the hard ceiling and both authorization phrases match exactly.

The repository is cloned at one resolved commit beneath `/content`, installed from that checkout, and discarded with the runtime. It is never cloned into Drive. Only the Archive IR, cost ledger, resumable semantic cache, and final outputs persist in the project Drive folder.

In [ ]:
import os
import re
import subprocess
import sys
import tempfile
from collections.abc import Iterator, Mapping, Sequence
from contextlib import contextmanager


def run_command(
    command: Sequence[str],
    *,
    cwd: Path | None = None,
    env: Mapping[str, str] | None = None,
    capture_output: bool = False,
    check: bool = True,
) -> subprocess.CompletedProcess[str]:
    """Run one shell-free subprocess with explicit arguments."""

    return subprocess.run(
        list(command),
        cwd=cwd,
        env=dict(env) if env is not None else None,
        check=check,
        text=True,
        capture_output=capture_output,
    )


def get_colab_secret(name: str) -> str:
    """Read one named secret without exposing provider error text."""

    try:
        from google.colab import userdata

        value = userdata.get(name)
    except Exception:
        raise RuntimeError(
            f"Colab secret {name} is missing or Notebook access is disabled."
        ) from None
    if not value:
        raise RuntimeError(f"Colab secret {name} is empty.")
    return value


ASKPASS_SOURCE = """#!/usr/bin/env python3
import os
import sys

prompt = sys.argv[1].lower() if len(sys.argv) > 1 else ""
print("x-access-token" if "username" in prompt else os.environ["CAC_GIT_TOKEN"])
"""


@contextmanager
def authenticated_git_environment() -> Iterator[dict[str, str]]:
    """Yield a subprocess environment backed by a temporary askpass helper."""

    token = get_colab_secret("GITHUB_TOKEN")
    with tempfile.TemporaryDirectory(prefix="cac_git_auth_", dir="/content") as directory:
        helper = Path(directory) / "askpass.py"
        helper.write_text(ASKPASS_SOURCE, encoding="utf-8")
        helper.chmod(0o700)
        environment = os.environ.copy()
        environment.pop("GITHUB_TOKEN", None)
        environment.update(
            {
                "CAC_GIT_TOKEN": token,
                "GIT_ASKPASS": str(helper),
                "GIT_TERMINAL_PROMPT": "0",
            }
        )
        try:
            yield environment
        finally:
            environment.pop("CAC_GIT_TOKEN", None)


def resolve_remote_commit() -> str:
    """Resolve the selected branch to exactly one 40-character Git SHA."""

    if (
        run_command(
            ["git", "check-ref-format", "--branch", REPO_BRANCH],
            capture_output=True,
            check=False,
        ).returncode
        != 0
    ):
        raise ValueError("REPO_BRANCH is not a valid Git branch name.")

    expected_ref = f"refs/heads/{REPO_BRANCH}"
    with authenticated_git_environment() as environment:
        result = run_command(
            [
                "git",
                "-c",
                "credential.helper=",
                "ls-remote",
                "--exit-code",
                PUBLIC_REPO_URL,
                expected_ref,
            ],
            env=environment,
            capture_output=True,
        )
    matches = [
        fields[0]
        for line in result.stdout.splitlines()
        if len(fields := line.split()) == 2 and fields[1] == expected_ref
    ]
    if len(matches) != 1 or re.fullmatch(r"[0-9a-f]{40}", matches[0]) is None:
        raise RuntimeError("Selected branch did not resolve to exactly one Git commit.")
    return matches[0]

In [ ]:
import importlib
import inspect

CHECKED_OUT_COMMIT = resolve_remote_commit()
REPO_DIR = Path(
    tempfile.mkdtemp(
        prefix=f"chatgpt-archive-compiler-{CHECKED_OUT_COMMIT[:12]}-",
        dir="/content",
    )
)
with authenticated_git_environment() as environment:
    run_command(
        [
            "git",
            "-c",
            "credential.helper=",
            "clone",
            "--branch",
            REPO_BRANCH,
            "--single-branch",
            "--no-tags",
            PUBLIC_REPO_URL,
            str(REPO_DIR),
        ],
        env=environment,
    )
run_command(["git", "checkout", "--detach", CHECKED_OUT_COMMIT], cwd=REPO_DIR)
actual_commit = run_command(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True
).stdout.strip()
if actual_commit != CHECKED_OUT_COMMIT:
    raise RuntimeError("Ephemeral checkout did not resolve to the selected commit.")

run_command(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "-e",
        f"{REPO_DIR}[notebooks,pdf,semantic]",
    ]
)
repository_source = (REPO_DIR / "src").resolve()
sys.path.insert(0, str(repository_source))
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "chatgpt_archive_compiler" or module_name.startswith(
        "chatgpt_archive_compiler."
    ):
        del sys.modules[module_name]

archive_compiler = importlib.import_module("chatgpt_archive_compiler")
ingest_module = importlib.import_module("chatgpt_archive_compiler.ingest")
semantic_module = importlib.import_module("chatgpt_archive_compiler.semantic")
semantic_book_module = importlib.import_module("chatgpt_archive_compiler.semantic_book")
serialization_module = importlib.import_module("chatgpt_archive_compiler.serialization")

IngestLimits = ingest_module.IngestLimits
SchemaMode = ingest_module.SchemaMode
ingest_export_zip = ingest_module.ingest_export_zip
SemanticAtlasOptions = semantic_module.SemanticAtlasOptions
ApiBudget = semantic_module.ApiBudget
ModelTokenPrice = semantic_module.ModelTokenPrice
RoutedStructuredAnalysisProvider = semantic_module.RoutedStructuredAnalysisProvider
estimate_budgeted_semantic_cost = semantic_module.estimate_budgeted_semantic_cost
OpenAIEmbeddingProvider = semantic_module.OpenAIEmbeddingProvider
OpenAIStructuredAnalysisProvider = semantic_module.OpenAIStructuredAnalysisProvider
LocalHashingEmbeddingProvider = semantic_module.LocalHashingEmbeddingProvider
LocalHeuristicAnalysisProvider = semantic_module.LocalHeuristicAnalysisProvider
estimate_semantic_run = semantic_module.estimate_semantic_run
build_semantic_atlas = semantic_module.build_semantic_atlas
SemanticBookOptions = semantic_book_module.SemanticBookOptions
BookPaperSize = semantic_book_module.BookPaperSize
compile_semantic_book = semantic_book_module.compile_semantic_book
read_archive_ir = serialization_module.read_archive_ir
exceptions_module = importlib.import_module("chatgpt_archive_compiler.exceptions")
ApiBudgetExceededError = exceptions_module.ApiBudgetExceededError
SemanticAtlasError = semantic_module.SemanticAtlasError

imported_from = Path(archive_compiler.__file__).resolve()
if repository_source not in imported_from.parents:
    raise RuntimeError("Package import did not resolve to the ephemeral checkout.")
dirty = run_command(
    ["git", "status", "--porcelain", "--untracked-files=all"],
    cwd=REPO_DIR,
    capture_output=True,
).stdout
if dirty:
    raise RuntimeError("Package installation unexpectedly changed the Git checkout.")

print(f"Ephemeral checkout: {REPO_DIR}")
print(f"Commit: {CHECKED_OUT_COMMIT}")
print(f"Package version: {archive_compiler.__version__}")
print(f"Imported from: {imported_from}")

## Analytical model

The atlas keeps **subject**, **project**, **purpose**, and **recurring theme** separate. Its cost-controlled evidence flow is:

1. Construct bounded current-path user/assistant representations locally.
2. Embed every conversation and construct the similarity graph.
3. Create transparent local profiles for every conversation.
4. Discover and, only when necessary, consolidate graph communities.
5. Select central, earliest, and latest representatives fairly across categories.
6. Refine no more than `MAX_REFINED_CONVERSATIONS` with the profile model.
7. Name categories from compact community dossiers.
8. Perform one final synthesis from representative profiles, category structure, and bounded project evidence.

Thus every chat participates in the geometry and final catalog, but API-priced generative work scales with the number of categories rather than the number of conversations. Separate caches preserve embeddings, local profiles, refined profiles, taxonomy, and synthesis without confusing baseline and model-refined records.

In [ ]:
from collections.abc import Callable
from typing import Any


def construct_supported(model_type: type[Any], values: Mapping[str, Any]) -> Any:
    """Construct a package option/provider object using its declared fields only."""

    model_fields = getattr(model_type, "model_fields", None)
    if isinstance(model_fields, Mapping):
        accepted = set(model_fields)
    else:
        signature = inspect.signature(model_type)
        if any(
            parameter.kind is inspect.Parameter.VAR_KEYWORD
            for parameter in signature.parameters.values()
        ):
            accepted = set(values)
        else:
            accepted = set(signature.parameters)
    return model_type(**{key: value for key, value in values.items() if key in accepted})


def call_supported(function: Callable[..., Any], /, *args: Any, **kwargs: Any) -> Any:
    """Call a public package API while tolerating additive optional parameters."""

    signature = inspect.signature(function)
    if any(
        parameter.kind is inspect.Parameter.VAR_KEYWORD
        for parameter in signature.parameters.values()
    ):
        return function(*args, **kwargs)
    accepted = set(signature.parameters)
    return function(*args, **{key: value for key, value in kwargs.items() if key in accepted})


def semantic_progress(stage: str, completed: int, total: int, cached: int) -> None:
    """Print content-free progress for long resumable semantic stages."""

    allowed_stages = {
        "representations",
        "embeddings",
        "conversation_profiles",
        "representative_profiles",
        "semantic_graph",
        "taxonomy",
        "archive_synthesis",
        "artifact_export",
    }
    safe_stage = stage if stage in allowed_stages else "semantic_stage"
    cache_note = f"; {cached:,} cached" if cached else ""
    print(f"[{safe_stage}] {completed:,}/{total:,}{cache_note}")


def make_atlas_options(*, mode: str = ANALYSIS_MODE) -> Any:
    """Build version-compatible semantic options from notebook controls."""

    return construct_supported(
        SemanticAtlasOptions,
        {
            "mode": mode,
            "analysis_mode": mode,
            "analysis_model": PROFILE_MODEL,
            "model": PROFILE_MODEL,
            "embedding_model": EMBEDDING_MODEL,
            "embedding_dimensions": EMBEDDING_DIMENSIONS,
            "reasoning_effort": PROFILE_REASONING_EFFORT,
            "max_characters_per_conversation": MAX_CHARACTERS_PER_CONVERSATION,
            "embedding_batch_size": EMBEDDING_BATCH_SIZE,
            "analysis_batch_size": ANALYSIS_BATCH_SIZE,
            "max_neighbors": NEAREST_NEIGHBORS,
            "nearest_neighbors": NEAREST_NEIGHBORS,
            "min_similarity": MIN_SIMILARITY,
            "max_leaf_categories": MAX_LEAF_CATEGORIES,
            "refined_conversations_per_category": REFINED_CONVERSATIONS_PER_CATEGORY,
            "max_refined_conversations": (MAX_REFINED_CONVERSATIONS if mode == "budgeted" else 0),
            "review_confidence_threshold": REVIEW_CONFIDENCE_THRESHOLD,
            "cache_enabled": True,
            "include_reasoning": False,
            "max_conversations": MAX_CONVERSATIONS or None,
        },
    )


SAFE_AGGREGATE_FIELDS = {
    "conversation_count",
    "selected_conversation_count",
    "message_count",
    "character_count",
    "original_character_count",
    "approximate_input_tokens",
    "embedding_item_count",
    "analysis_item_count",
    "truncated_conversation_count",
    "estimated_input_tokens",
    "estimated_output_tokens",
    "estimated_embedding_tokens",
    "estimated_total_tokens",
    "estimated_request_count",
    "estimated_batch_count",
    "estimated_cost_usd",
    "locally_profiled_conversation_count",
    "model_refined_conversation_count",
    "maximum_leaf_category_count",
    "profile_request_count",
    "estimated_embedding_input_tokens",
    "estimated_structured_input_tokens",
    "estimated_structured_output_tokens",
    "expected_cost_usd",
    "conservative_scheduled_reserve_usd",
    "hard_budget_usd",
    "scheduled_plan_fits_hard_budget",
    "charged_cost_usd",
    "remaining_cost_usd",
    "request_count",
    "completed_request_count",
    "failed_or_unsettled_request_count",
    "category_count",
    "subcategory_count",
    "theme_count",
    "project_count",
    "edge_count",
    "review_item_count",
    "cached_item_count",
    "computed_item_count",
    "unclassified_conversation_count",
}


def safe_aggregate_summary(value: Any) -> dict[str, int | float | str | None]:
    """Return only explicitly approved, non-content scalar fields."""

    if hasattr(value, "model_dump"):
        raw = value.model_dump(mode="json")
    elif isinstance(value, Mapping):
        raw = dict(value)
    else:
        raw = vars(value) if hasattr(value, "__dict__") else {}
    return {
        key: item
        for key, item in raw.items()
        if key in SAFE_AGGREGATE_FIELDS and (item is None or isinstance(item, (int, float, str)))
    }

## Synthetic full-pipeline and budget-plan validation — no API calls

This test creates two deliberately related conversations, ingests them through the real Archive IR path, completes the entire semantic pipeline with local providers, renders an actual PDF, and verifies the budget planner independently. It validates package imports, schema compatibility, artifact export, and typesetting without retrieving an OpenAI key or sending any data externally.

In [ ]:
import json
import zipfile


def synthetic_conversation(
    conversation_id: str, title: str, timestamp: int, question: str, answer: str
) -> dict[str, object]:
    """Create one minimal export-shaped conversation for package validation."""

    return {
        "id": conversation_id,
        "title": title,
        "create_time": timestamp,
        "update_time": timestamp + 60,
        "current_node": f"{conversation_id}-assistant",
        "mapping": {
            f"{conversation_id}-root": {
                "id": f"{conversation_id}-root",
                "parent": None,
                "message": None,
            },
            f"{conversation_id}-user": {
                "id": f"{conversation_id}-user",
                "parent": f"{conversation_id}-root",
                "message": {
                    "id": f"{conversation_id}-message-user",
                    "author": {"role": "user"},
                    "create_time": timestamp,
                    "content": {"content_type": "text", "parts": [question]},
                    "metadata": {},
                },
            },
            f"{conversation_id}-assistant": {
                "id": f"{conversation_id}-assistant",
                "parent": f"{conversation_id}-user",
                "message": {
                    "id": f"{conversation_id}-message-assistant",
                    "author": {"role": "assistant"},
                    "create_time": timestamp + 60,
                    "content": {"content_type": "text", "parts": [answer]},
                    "metadata": {},
                },
            },
        },
    }


synthetic_payload = [
    synthetic_conversation(
        "semantic-one",
        "Synthetic representation-learning study",
        1_735_689_600,
        "Design a reproducible representation-learning comparison.",
        "Define datasets, held-out validation, metrics, and provenance.",
    ),
    synthetic_conversation(
        "semantic-two",
        "Synthetic validation follow-up",
        1_738_368_000,
        "How should the earlier comparison be externally validated?",
        "Preserve the split and test transfer on an independent dataset.",
    ),
]

with tempfile.TemporaryDirectory(prefix="cac_semantic_smoke_", dir="/content") as directory:
    temporary_path = Path(directory)
    synthetic_zip = temporary_path / "synthetic-export.zip"
    with zipfile.ZipFile(synthetic_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive_zip:
        archive_zip.writestr("conversations.json", json.dumps(synthetic_payload, allow_nan=False))
    synthetic_archive = ingest_export_zip(
        synthetic_zip, limits=IngestLimits(), schema_mode=SchemaMode.STRICT
    )
    synthetic_options = make_atlas_options(mode="local")
    synthetic_estimate = call_supported(
        estimate_semantic_run,
        synthetic_archive,
        options=synthetic_options,
    )
    synthetic_summary = safe_aggregate_summary(synthetic_estimate)
    synthetic_cost_plan = estimate_budgeted_semantic_cost(
        synthetic_estimate,
        embedding_price=ModelTokenPrice(input_usd_per_million=EMBEDDING_INPUT_USD_PER_MILLION),
        profile_price=ModelTokenPrice(
            input_usd_per_million=PROFILE_INPUT_USD_PER_MILLION,
            output_usd_per_million=PROFILE_OUTPUT_USD_PER_MILLION,
        ),
        taxonomy_price=ModelTokenPrice(
            input_usd_per_million=TAXONOMY_INPUT_USD_PER_MILLION,
            output_usd_per_million=TAXONOMY_OUTPUT_USD_PER_MILLION,
        ),
        synthesis_price=ModelTokenPrice(
            input_usd_per_million=SYNTHESIS_INPUT_USD_PER_MILLION,
            output_usd_per_million=SYNTHESIS_OUTPUT_USD_PER_MILLION,
        ),
        hard_budget_usd=HARD_API_BUDGET_USD,
        max_leaf_categories=MAX_LEAF_CATEGORIES,
        refined_conversations_per_category=REFINED_CONVERSATIONS_PER_CATEGORY,
        max_refined_conversations=MAX_REFINED_CONVERSATIONS,
        analysis_batch_size=ANALYSIS_BATCH_SIZE,
        embedding_max_tokens_per_item=EMBEDDING_MAX_TOKENS_PER_ITEM,
        profile_max_input_tokens_per_item=EMBEDDING_MAX_TOKENS_PER_ITEM,
        profile_max_output_tokens_per_batch=PROFILE_MAX_OUTPUT_TOKENS,
        taxonomy_max_output_tokens=TAXONOMY_MAX_OUTPUT_TOKENS,
        synthesis_max_output_tokens=SYNTHESIS_MAX_OUTPUT_TOKENS,
    )
    assert synthetic_cost_plan.scheduled_plan_fits_hard_budget
    estimated_conversations = synthetic_summary.get(
        "selected_conversation_count", synthetic_summary.get("conversation_count")
    )
    if estimated_conversations is not None:
        assert estimated_conversations == 2
    synthetic_result = call_supported(
        build_semantic_atlas,
        synthetic_archive,
        temporary_path / "atlas",
        embedding_provider=construct_supported(LocalHashingEmbeddingProvider, {}),
        analysis_provider=construct_supported(LocalHeuristicAnalysisProvider, {}),
        options=synthetic_options,
    )
    synthetic_book_options = construct_supported(
        SemanticBookOptions,
        {
            "title": "Synthetic Semantic Atlas",
            "paper_size": BookPaperSize.TRADE,
            "include_transcripts": False,
            "render_pdf": True,
        },
    )
    synthetic_book_result = call_supported(
        compile_semantic_book,
        synthetic_archive,
        synthetic_result.atlas,
        temporary_path / "book",
        options=synthetic_book_options,
    )
    assert synthetic_result.catalog_path.is_file()
    assert synthetic_result.graph_path.is_file()
    assert synthetic_result.taxonomy_path.is_file()
    assert synthetic_result.manifest_path.is_file()
    assert synthetic_result.atlas_html_path.is_file()
    assert synthetic_book_result.html_path.is_file()
    assert synthetic_book_result.pdf_path is not None
    assert synthetic_book_result.pdf_path.read_bytes().startswith(b"%PDF")
    assert synthetic_book_result.manifest_path.is_file()

print("Synthetic Archive-IR-to-atlas-to-book validation passed without API access.")
for key, value in sorted(synthetic_summary.items()):
    print(f"{key}: {value}")

## Real archive preflight — local, resumable, and cost-gated

Leave `ARCHIVE_IR_PATH` blank to select the newest successful `archive.ir.json` produced by notebook 01, or supply an exact path. Preflight reads that IR locally and reports aggregate volume, expected cost, and the more conservative scheduled reservation. It does **not** retrieve `OPENAI_API_KEY`, instantiate an external client, or send data to any API.

The plan counts every conversation as locally profiled but caps model-refined representatives at `MAX_REFINED_CONVERSATIONS`. If the full scheduled reservation does not fit `HARD_API_BUDGET_USD`, preflight stops before the authorization cell becomes usable.

For an interrupted run, set `RESUME_OUTPUT_DIRECTORY` to its exact prior directory. Archive checksum, commit, models, prices, limits, and hard budget must all match. The persistent `api_budget_ledger.json` preserves charged or conservatively reserved cost across runtime restarts. This revision explicitly permits the cache-compatible migration from the two earlier budgeted-workflow commits affected by the embedding batching limit; it records that migration without discarding completed embeddings.

In [ ]:
import hashlib
from datetime import UTC, datetime

PREPARE_REAL_ANALYSIS = False  # @param {type:"boolean"}
ARCHIVE_IR_PATH = ""  # @param {type:"string"}
RESUME_OUTPUT_DIRECTORY = ""  # @param {type:"string"}
COLAB_PRIVACY_ACKNOWLEDGEMENT = ""  # @param {type:"string"}
REQUIRED_COLAB_ACKNOWLEDGEMENT = "I UNDERSTAND THIS READS MY ARCHIVE IN GOOGLE COLAB"


def sha256_file(path: Path) -> str:
    """Hash one file incrementally."""

    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()


def configured_prices() -> dict[str, object]:
    """Construct explicit price objects shared by planning and live authorization."""

    return {
        "embedding": ModelTokenPrice(input_usd_per_million=EMBEDDING_INPUT_USD_PER_MILLION),
        "profile": ModelTokenPrice(
            input_usd_per_million=PROFILE_INPUT_USD_PER_MILLION,
            output_usd_per_million=PROFILE_OUTPUT_USD_PER_MILLION,
        ),
        "taxonomy": ModelTokenPrice(
            input_usd_per_million=TAXONOMY_INPUT_USD_PER_MILLION,
            output_usd_per_million=TAXONOMY_OUTPUT_USD_PER_MILLION,
        ),
        "synthesis": ModelTokenPrice(
            input_usd_per_million=SYNTHESIS_INPUT_USD_PER_MILLION,
            output_usd_per_million=SYNTHESIS_OUTPUT_USD_PER_MILLION,
        ),
    }


PREPARED_RUN = None
if not PREPARE_REAL_ANALYSIS:
    print("Real semantic preflight remains disabled.")
else:
    if COLAB_PRIVACY_ACKNOWLEDGEMENT != REQUIRED_COLAB_ACKNOWLEDGEMENT:
        raise RuntimeError("Exact Colab privacy acknowledgment is required.")
    if ARCHIVE_IR_PATH.strip():
        archive_ir_path = Path(ARCHIVE_IR_PATH).expanduser().resolve()
    else:
        compiled_output_root = DRIVE_PROJECT_DIR / "outputs" / "compiled"
        candidates = sorted(
            path.resolve()
            for path in compiled_output_root.glob("*/archive.ir.json")
            if (path.parent / "compilation_manifest.json").is_file()
        )
        if not candidates:
            raise RuntimeError(
                "No successful compiled archive was found; provide ARCHIVE_IR_PATH explicitly."
            )
        newest_mtime_ns = max(path.stat().st_mtime_ns for path in candidates)
        newest_candidates = [
            path for path in candidates if path.stat().st_mtime_ns == newest_mtime_ns
        ]
        if len(newest_candidates) != 1:
            raise RuntimeError(
                "Multiple equally recent compiled archives were found; provide "
                "ARCHIVE_IR_PATH explicitly."
            )
        archive_ir_path = newest_candidates[0]
        print(f"Auto-selected Archive IR: {archive_ir_path}")
    if not archive_ir_path.is_file() or archive_ir_path.name != "archive.ir.json":
        raise RuntimeError("ARCHIVE_IR_PATH must identify an existing archive.ir.json file.")

    output_root_resolved = OUTPUT_ROOT.resolve()
    if RESUME_OUTPUT_DIRECTORY.strip():
        semantic_output_directory = Path(RESUME_OUTPUT_DIRECTORY).expanduser().resolve()
        if not semantic_output_directory.is_dir():
            raise RuntimeError("RESUME_OUTPUT_DIRECTORY must identify an existing directory.")
        if not semantic_output_directory.is_relative_to(output_root_resolved):
            raise RuntimeError("Resume directory must remain inside the semantic output root.")
    else:
        run_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
        semantic_output_directory = OUTPUT_ROOT / f"{run_id}-{CHECKED_OUT_COMMIT[:12]}-budgeted"
        semantic_output_directory.mkdir(parents=True, exist_ok=False)

    archive_sha256 = sha256_file(archive_ir_path)
    atlas_options = make_atlas_options()
    plan_identity = {
        "archive_sha256": archive_sha256,
        "repository_commit": CHECKED_OUT_COMMIT,
        "analysis_mode": ANALYSIS_MODE,
        "embedding_model": EMBEDDING_MODEL,
        "embedding_dimensions": EMBEDDING_DIMENSIONS,
        "embedding_max_tokens_per_item": EMBEDDING_MAX_TOKENS_PER_ITEM,
        "profile_model": PROFILE_MODEL,
        "taxonomy_model": TAXONOMY_MODEL,
        "synthesis_model": SYNTHESIS_MODEL,
        "profile_reasoning_effort": PROFILE_REASONING_EFFORT,
        "taxonomy_reasoning_effort": TAXONOMY_REASONING_EFFORT,
        "synthesis_reasoning_effort": SYNTHESIS_REASONING_EFFORT,
        "profile_max_output_tokens": PROFILE_MAX_OUTPUT_TOKENS,
        "taxonomy_max_output_tokens": TAXONOMY_MAX_OUTPUT_TOKENS,
        "synthesis_max_output_tokens": SYNTHESIS_MAX_OUTPUT_TOKENS,
        "max_characters_per_conversation": MAX_CHARACTERS_PER_CONVERSATION,
        "embedding_batch_size": EMBEDDING_BATCH_SIZE,
        "embedding_tokens_per_minute": EMBEDDING_TOKENS_PER_MINUTE,
        "analysis_batch_size": ANALYSIS_BATCH_SIZE,
        "nearest_neighbors": NEAREST_NEIGHBORS,
        "min_similarity": MIN_SIMILARITY,
        "max_leaf_categories": MAX_LEAF_CATEGORIES,
        "refined_conversations_per_category": REFINED_CONVERSATIONS_PER_CATEGORY,
        "max_refined_conversations": MAX_REFINED_CONVERSATIONS,
        "review_confidence_threshold": REVIEW_CONFIDENCE_THRESHOLD,
        "max_conversations": MAX_CONVERSATIONS or None,
        "price_snapshot_date": PRICE_SNAPSHOT_DATE,
        "prices": {
            "embedding_input": EMBEDDING_INPUT_USD_PER_MILLION,
            "profile_input": PROFILE_INPUT_USD_PER_MILLION,
            "profile_output": PROFILE_OUTPUT_USD_PER_MILLION,
            "taxonomy_input": TAXONOMY_INPUT_USD_PER_MILLION,
            "taxonomy_output": TAXONOMY_OUTPUT_USD_PER_MILLION,
            "synthesis_input": SYNTHESIS_INPUT_USD_PER_MILLION,
            "synthesis_output": SYNTHESIS_OUTPUT_USD_PER_MILLION,
        },
        "hard_api_budget_usd": HARD_API_BUDGET_USD,
    }
    plan_identity_path = semantic_output_directory / "run_identity.json"
    if plan_identity_path.exists():
        existing_identity = json.loads(plan_identity_path.read_text(encoding="utf-8"))
        if existing_identity != plan_identity:
            if not isinstance(existing_identity, dict):
                raise RuntimeError("Existing resume identity is not a JSON object.")
            source_commit = existing_identity.get("repository_commit")
            existing_comparable = dict(existing_identity)
            planned_comparable = dict(plan_identity)
            existing_comparable.pop("repository_commit", None)
            planned_comparable.pop("repository_commit", None)
            existing_comparable.setdefault(
                "embedding_tokens_per_minute", EMBEDDING_TOKENS_PER_MINUTE
            )
            if (
                source_commit not in SAFE_RESUME_SOURCE_COMMITS
                or existing_comparable != planned_comparable
            ):
                raise RuntimeError(
                    "Resume identity does not match this archive, budget, or configuration."
                )
            migration_record = {
                "from_repository_commit": source_commit,
                "to_repository_commit": CHECKED_OUT_COMMIT,
                "reason": "cache-compatible embedding request-limit and pacing fix",
            }
            (semantic_output_directory / "resume_migration.json").write_text(
                json.dumps(migration_record, indent=2, sort_keys=True) + "\n",
                encoding="utf-8",
            )
            plan_identity_path.write_text(
                json.dumps(plan_identity, indent=2, sort_keys=True, allow_nan=False) + "\n",
                encoding="utf-8",
            )
            print("Accepted the cache-compatible embedding resume migration.")
    else:
        plan_identity_path.write_text(
            json.dumps(plan_identity, indent=2, sort_keys=True, allow_nan=False) + "\n",
            encoding="utf-8",
        )

    try:
        real_archive = read_archive_ir(archive_ir_path)
        real_estimate = call_supported(estimate_semantic_run, real_archive, options=atlas_options)
        prices = configured_prices()
        cost_plan = estimate_budgeted_semantic_cost(
            real_estimate,
            embedding_price=prices["embedding"],
            profile_price=prices["profile"],
            taxonomy_price=prices["taxonomy"],
            synthesis_price=prices["synthesis"],
            hard_budget_usd=HARD_API_BUDGET_USD,
            max_leaf_categories=MAX_LEAF_CATEGORIES,
            refined_conversations_per_category=REFINED_CONVERSATIONS_PER_CATEGORY,
            max_refined_conversations=MAX_REFINED_CONVERSATIONS,
            analysis_batch_size=ANALYSIS_BATCH_SIZE,
            embedding_max_tokens_per_item=EMBEDDING_MAX_TOKENS_PER_ITEM,
            profile_max_input_tokens_per_item=EMBEDDING_MAX_TOKENS_PER_ITEM,
            profile_max_output_tokens_per_batch=PROFILE_MAX_OUTPUT_TOKENS,
            taxonomy_max_output_tokens=TAXONOMY_MAX_OUTPUT_TOKENS,
            synthesis_max_output_tokens=SYNTHESIS_MAX_OUTPUT_TOKENS,
        )
    except Exception as exception:
        raise RuntimeError(
            f"Semantic preflight failed safely ({type(exception).__name__}); "
            "source-derived exception text was suppressed."
        ) from None

    estimate_summary = safe_aggregate_summary(real_estimate)
    cost_plan_summary = safe_aggregate_summary(cost_plan)
    (semantic_output_directory / "preflight_estimate.json").write_text(
        json.dumps(
            {
                "semantic_run": estimate_summary,
                "budgeted_cost_plan": cost_plan.model_dump(mode="json"),
                "price_snapshot_date": PRICE_SNAPSHOT_DATE,
            },
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )
    if ANALYSIS_MODE == "budgeted" and not cost_plan.scheduled_plan_fits_hard_budget:
        raise RuntimeError(
            "The conservative scheduled reservation exceeds HARD_API_BUDGET_USD. "
            "Reduce categories, refined conversations, output ceilings, or model prices."
        )
    PREPARED_RUN = {
        "archive": real_archive,
        "archive_sha256": archive_sha256,
        "atlas_options": atlas_options,
        "output_directory": semantic_output_directory,
        "estimate_summary": estimate_summary,
        "cost_plan": cost_plan,
        "prices": prices,
    }

    print("Semantic preflight completed without external API access.")
    for key, value in sorted(estimate_summary.items()):
        print(f"semantic_run.{key}: {value}")
    for key, value in sorted(cost_plan_summary.items()):
        print(f"budgeted_cost_plan.{key}: {value}")
    print(f"Price snapshot: {PRICE_SNAPSHOT_DATE}")
    print(f"Prepared output directory: {semantic_output_directory}")
    print("The authorized cell remains disabled until both acknowledgments match exactly.")

## Authorized budgeted atlas construction and book rendering

Embedding sends every bounded title/date/current-path user-assistant representation to the configured embeddings endpoint. Structured model analysis sends full bounded text only for the selected representatives—at most `MAX_REFINED_CONVERSATIONS`—then sends compact profile/category dossiers for taxonomy and final synthesis. Source IDs and paths, warning locations, reasoning traces, system/tool messages, alternate branches, and attachment contents or locations remain excluded.

Responses requests set `store=False`. This prevents Responses application-state storage, but default abuse-monitoring logs may retain prompts and responses for up to 30 days; API data is not used for training unless the account opts in. The privacy acknowledgment and the exact dollar authorization are separate.

`api_budget_ledger.json` is written before each external request. A request is refused if its conservative reservation would exceed the remaining configured budget. If a response reports usage, the reservation is reduced to actual token cost; failed or unobservable requests retain their reservation. SDK-level automatic retries are disabled, so every later transmission must pass the ledger independently. Embeddings are repartitioned below the API's aggregate 300,000-token request limit and conservatively paced below the Tier 1 tokens-per-minute limit; pauses of up to roughly one minute are intentional. The ledger contains stages, models, token counts, prices, and costs—never archive text.

In [ ]:
RUN_REAL_ANALYSIS = False  # @param {type:"boolean"}
EXTERNAL_MODEL_ACKNOWLEDGEMENT = ""  # @param {type:"string"}
BUDGET_ACKNOWLEDGEMENT = ""  # @param {type:"string"}
REQUIRED_EXTERNAL_ACKNOWLEDGEMENT = (
    "I AUTHORIZE OPENAI PROCESSING AND ACCEPT UP TO 30 DAYS OF ABUSE-MONITORING RETENTION"
)
REQUIRED_BUDGET_ACKNOWLEDGEMENT = (
    f"I AUTHORIZE A MAXIMUM CONFIGURED API COST OF ${HARD_API_BUDGET_USD:.2f}"
)

if not RUN_REAL_ANALYSIS:
    print("Real semantic analysis remains disabled.")
    print(f"Required budget phrase: {REQUIRED_BUDGET_ACKNOWLEDGEMENT}")
else:
    if PREPARED_RUN is None:
        raise RuntimeError("Run the enabled real-archive preflight cell first.")
    if ANALYSIS_MODE == "budgeted":
        if EXTERNAL_MODEL_ACKNOWLEDGEMENT != REQUIRED_EXTERNAL_ACKNOWLEDGEMENT:
            raise RuntimeError("Exact external-processing acknowledgment is required.")
        if BUDGET_ACKNOWLEDGEMENT != REQUIRED_BUDGET_ACKNOWLEDGEMENT:
            raise RuntimeError("Exact configured-budget acknowledgment is required.")
        if not PREPARED_RUN["cost_plan"].scheduled_plan_fits_hard_budget:
            raise RuntimeError("Prepared cost plan no longer fits the configured hard budget.")

    baseline_provider = LocalHeuristicAnalysisProvider()
    api_budget = None
    refinement_provider = None
    interpretation_provider = baseline_provider
    if ANALYSIS_MODE == "budgeted":
        api_budget = ApiBudget(
            max_cost_usd=HARD_API_BUDGET_USD,
            ledger_path=PREPARED_RUN["output_directory"] / "api_budget_ledger.json",
        )
        openai_api_key = get_colab_secret("OPENAI_API_KEY")
        embedding_provider = OpenAIEmbeddingProvider(
            api_key=openai_api_key,
            max_retries=0,
            model=EMBEDDING_MODEL,
            dimensions=EMBEDDING_DIMENSIONS,
            request_batch_size=EMBEDDING_BATCH_SIZE,
            maximum_tokens_per_item=EMBEDDING_MAX_TOKENS_PER_ITEM,
            tokens_per_minute=EMBEDDING_TOKENS_PER_MINUTE,
            budget=api_budget,
            token_price=PREPARED_RUN["prices"]["embedding"],
        )
        profile_provider = OpenAIStructuredAnalysisProvider(
            api_key=openai_api_key,
            max_retries=0,
            model=PROFILE_MODEL,
            reasoning_effort=PROFILE_REASONING_EFFORT,
            synthesis_reasoning_effort=PROFILE_REASONING_EFFORT,
            profile_max_input_tokens_per_item=EMBEDDING_MAX_TOKENS_PER_ITEM,
            profile_max_output_tokens=PROFILE_MAX_OUTPUT_TOKENS,
            taxonomy_max_output_tokens=TAXONOMY_MAX_OUTPUT_TOKENS,
            synthesis_max_output_tokens=SYNTHESIS_MAX_OUTPUT_TOKENS,
            budget=api_budget,
            token_price=PREPARED_RUN["prices"]["profile"],
        )
        taxonomy_provider = OpenAIStructuredAnalysisProvider(
            api_key=openai_api_key,
            max_retries=0,
            model=TAXONOMY_MODEL,
            reasoning_effort=TAXONOMY_REASONING_EFFORT,
            synthesis_reasoning_effort=TAXONOMY_REASONING_EFFORT,
            profile_max_input_tokens_per_item=EMBEDDING_MAX_TOKENS_PER_ITEM,
            profile_max_output_tokens=PROFILE_MAX_OUTPUT_TOKENS,
            taxonomy_max_output_tokens=TAXONOMY_MAX_OUTPUT_TOKENS,
            synthesis_max_output_tokens=SYNTHESIS_MAX_OUTPUT_TOKENS,
            budget=api_budget,
            token_price=PREPARED_RUN["prices"]["taxonomy"],
        )
        synthesis_provider = OpenAIStructuredAnalysisProvider(
            api_key=openai_api_key,
            max_retries=0,
            model=SYNTHESIS_MODEL,
            reasoning_effort=SYNTHESIS_REASONING_EFFORT,
            synthesis_reasoning_effort=SYNTHESIS_REASONING_EFFORT,
            profile_max_input_tokens_per_item=EMBEDDING_MAX_TOKENS_PER_ITEM,
            profile_max_output_tokens=PROFILE_MAX_OUTPUT_TOKENS,
            taxonomy_max_output_tokens=TAXONOMY_MAX_OUTPUT_TOKENS,
            synthesis_max_output_tokens=SYNTHESIS_MAX_OUTPUT_TOKENS,
            budget=api_budget,
            token_price=PREPARED_RUN["prices"]["synthesis"],
        )
        routed_provider = RoutedStructuredAnalysisProvider(
            profile_provider=profile_provider,
            taxonomy_provider=taxonomy_provider,
            synthesis_provider=synthesis_provider,
        )
        refinement_provider = routed_provider
        interpretation_provider = routed_provider
        del openai_api_key
    else:
        embedding_provider = LocalHashingEmbeddingProvider()

    try:
        atlas_result = build_semantic_atlas(
            PREPARED_RUN["archive"],
            PREPARED_RUN["output_directory"],
            options=PREPARED_RUN["atlas_options"],
            embedding_provider=embedding_provider,
            analysis_provider=baseline_provider,
            refinement_provider=refinement_provider,
            interpretation_provider=interpretation_provider,
            progress_callback=semantic_progress,
        )
    except ApiBudgetExceededError as exception:
        raise RuntimeError(str(exception)) from None
    except SemanticAtlasError as exception:
        raise RuntimeError(str(exception)) from None
    except Exception as exception:
        raise RuntimeError(
            f"Semantic atlas construction failed safely ({type(exception).__name__}); "
            "source-derived exception text was suppressed. Cached stages and the cost "
            "ledger remain."
        ) from None

    if api_budget is not None:
        budget_summary = safe_aggregate_summary(api_budget.snapshot())
        for key, value in sorted(budget_summary.items()):
            print(f"api_budget.{key}: {value}")

    book_options = SemanticBookOptions(
        title=BOOK_TITLE,
        subtitle=BOOK_SUBTITLE or None,
        author=BOOK_AUTHOR or None,
        edition="First budgeted semantic edition",
        paper_size=BookPaperSize(PAPER_SIZE),
        render_pdf=RENDER_PDF,
        include_transcripts=False,
        include_message_timestamps=False,
        max_conversations=MAX_CONVERSATIONS or None,
        max_related_conversations=8,
        max_synopsis_entries_per_category=12,
        max_expanded_conversations=MAX_EXPANDED_BOOK_PROFILES,
        max_project_timelines=MAX_PROJECT_TIMELINES,
        max_timeline_events_per_project=MAX_TIMELINE_EVENTS_PER_PROJECT,
    )
    try:
        book_result = compile_semantic_book(
            PREPARED_RUN["archive"],
            atlas_result.atlas,
            PREPARED_RUN["output_directory"] / "book",
            options=book_options,
        )
    except Exception as exception:
        raise RuntimeError(
            f"Semantic book rendering failed safely ({type(exception).__name__}); "
            "source-derived exception text was suppressed. Atlas artifacts and cache remain."
        ) from None

    print("Budgeted semantic atlas and thematic book completed.")
    for attribute in (
        "catalog_path",
        "graph_path",
        "taxonomy_path",
        "category_profiles_path",
        "project_timelines_path",
        "synthesis_path",
        "review_queue_path",
        "atlas_html_path",
        "html_path",
        "pdf_path",
        "manifest_path",
    ):
        for label, result in (("atlas", atlas_result), ("book", book_result)):
            artifact_path = getattr(result, attribute, None)
            if artifact_path is not None:
                print(f"{label}.{attribute}: {artifact_path}")
    print(f"Output directory: {PREPARED_RUN['output_directory']}")

## Persistent deliverables and review loop

A completed run preserves the complete semantic catalog, similarity graph, hierarchical taxonomy, locally generated baseline profiles, bounded model-refined representative profiles, category profiles, project timelines, cross-archive synthesis, review queue, provenance manifest, persistent API cost ledger, caches, and HTML/PDF book under:

`MyDrive/ChatGPT Data Export/outputs/semantic/<timestamp>-<commit>-budgeted/`

The cost ledger is part of the reproducibility record. Do not delete it before resuming a partially completed run. A later optional deepening pass can target user-selected categories without repeating embeddings or the complete local baseline; it should receive its own explicit budget authorization.

This atlas remains a reviewable proposed organization rather than an irreversible verdict. No real archive or generated private artifact belongs in GitHub—only package code, tests, documentation, and this orchestration notebook are versioned there.